# FinGPT Resume Backtest: Track A (CPU-only)

This notebook is the lightweight recovery path for an existing `backtest_*.csv`.

What Track A does:
- load an existing backtest CSV
- re-apply Agent 2 PMI post-processing with a configurable `PMI_ALPHA`
- fetch realized returns with slow, single-threaded, throttled requests
- recompute metrics and save a new CSV

What Track A does not do:
- it does not re-run Agent 1
- it does not re-run Agent 2 model inference
- it is designed to run on Colab CPU


In [ ]:
# Cell 1 - Install lightweight dependencies
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

_pip("yfinance", "pandas", "numpy", "requests")
print("Dependencies installed.")


In [ ]:
# Cell 2 - Paths and knobs
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RESUME_CSV = "/content/drive/MyDrive/backtest_20260506T051302Z.csv"
OUT_DIR = "/content/drive/MyDrive"
REPO_DIR = "/content/drive/MyDrive/FinGPT_Part2"
PMI_PRIOR_PATH = os.path.join(REPO_DIR, "output/pmi_null_logprobs.json")

PMI_ALPHA = 1.0
CALIBRATION_T = 1.2
PRICE_FETCH_RETRIES = 3
PRICE_FETCH_SLEEP_SEC = 0.35
PRICE_FETCH_BATCH_SIZE = 20
PRICE_FETCH_BATCH_PAUSE_SEC = 2.0
REFRESH_EXISTING_PRICES = True

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Resume CSV    : {RESUME_CSV}")
print(f"Output dir    : {OUT_DIR}")
print(f"PMI prior path: {PMI_PRIOR_PATH}")
print(f"PMI alpha     : {PMI_ALPHA}")


In [ ]:
# Cell 3 - Load current backtest CSV and validate logits columns
import numpy as np
import pandas as pd

raw_df = pd.read_csv(RESUME_CSV)
print(f"Loaded {len(raw_df)} rows.")
print(f"Columns: {list(raw_df.columns)}")

required_raw_cols = [
    "raw_signal_logprob_A", "raw_signal_logprob_B", "raw_signal_logprob_C",
]
missing = [c for c in required_raw_cols if c not in raw_df.columns]
if missing:
    raise ValueError(f"Resume CSV is missing required raw logprob columns: {missing}")

raw_df["logits_list"] = raw_df.apply(
    lambda row: [row["raw_signal_logprob_A"], row["raw_signal_logprob_B"], row["raw_signal_logprob_C"]],
    axis=1,
)
has_logits = raw_df[required_raw_cols].notna().all(axis=1)
has_row_null = all(
    col in raw_df.columns for col in ["pmi_null_logprob_A", "pmi_null_logprob_B", "pmi_null_logprob_C"]
)

print(f"Rows with valid A/B/C logits : {has_logits.sum()} / {len(raw_df)}")
print(f"Rows without logits          : {(~has_logits).sum()} / {len(raw_df)}")
print(f"Per-row PMI null logprobs present: {has_row_null}")
print()
print("Original signal_direction distribution:")
if "signal_direction" in raw_df.columns:
    print(raw_df["signal_direction"].value_counts(dropna=False).to_string())
else:
    print("signal_direction column not found.")


In [ ]:
# Cell 4 - Offline PMI re-scoring with configurable alpha
import json

STRATEGY_SET = ["BUY", "HOLD", "SELL"]
_DIR_MAP = {0: "long", 1: "neutral", 2: "short"}
_POS_MAP = {"long": 1, "short": -1, "neutral": 0}

def _softmax(x, T=CALIBRATION_T):
    x = np.array(x, dtype=float) / T
    x -= x.max()
    e = np.exp(x)
    return e / e.sum()

def _load_pmi_prior(path: str):
    if not path or not os.path.exists(path):
        return None
    try:
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
        lp = data.get("null_logprobs")
        if isinstance(lp, list) and len(lp) == 3:
            return np.array(lp, dtype=float), data
    except Exception as exc:
        print(f"Warning: could not read PMI cache ({exc})")
    return None

logits_arr = np.array(raw_df.loc[has_logits, "logits_list"].tolist(), dtype=float)
_disk = _load_pmi_prior(PMI_PRIOR_PATH)

if has_row_null:
    null_arr = raw_df.loc[has_logits, ["pmi_null_logprob_A", "pmi_null_logprob_B", "pmi_null_logprob_C"]].to_numpy(dtype=float)
    pmi_source = "per-row pmi_null_logprob_* columns from CSV"
    print(f"PMI prior source : {pmi_source}")
elif _disk is not None:
    null_lp, _meta = _disk
    null_arr = np.repeat(null_lp.reshape(1, 3), len(logits_arr), axis=0)
    pmi_source = "disk cache (true model prior)"
    print(f"PMI prior source : {pmi_source}")
    print(f"  path           : {PMI_PRIOR_PATH}")
    print(f"  decision_prefix: {_meta.get('decision_prefix')}")
    print(f"  score_tokens   : {_meta.get('score_tokens')}")
    print(f"  computed_at    : {_meta.get('computed_at')}")
else:
    null_lp = logits_arr.mean(axis=0)
    null_arr = np.repeat(null_lp.reshape(1, 3), len(logits_arr), axis=0)
    pmi_source = "empirical mean (fallback)"
    print(f"PMI prior source : {pmi_source}")

print(f"PMI alpha = {PMI_ALPHA}")
pmi_arr = logits_arr - PMI_ALPHA * null_arr

corrected = raw_df.copy()
new_rows = []
for df_idx, logits_pmi in zip(raw_df.index[has_logits], pmi_arr):
    probs = _softmax(logits_pmi)
    best = int(probs.argmax())
    direction = _DIR_MAP[best]
    confidence = round(float(probs[best]), 6)
    realized = raw_df.loc[df_idx, "realized_return"] if "realized_return" in raw_df.columns else np.nan
    realized = float(realized) if pd.notna(realized) else np.nan
    position = _POS_MAP[direction] if pd.notna(realized) else np.nan
    strategy_return = position * realized if pd.notna(realized) else np.nan
    new_rows.append({
        "_df_idx": df_idx,
        "direction": direction,
        "confidence": confidence,
        "signal_direction": direction,
        "signal_confidence": confidence,
        "signal_prob_A": round(float(probs[0]), 6),
        "signal_prob_B": round(float(probs[1]), 6),
        "signal_prob_C": round(float(probs[2]), 6),
        "pmi_adjusted_logit_A": float(logits_pmi[0]),
        "pmi_adjusted_logit_B": float(logits_pmi[1]),
        "pmi_adjusted_logit_C": float(logits_pmi[2]),
        "pmi_alpha_used": PMI_ALPHA,
        "position": position,
        "strategy_return": strategy_return,
    })

pmi_df = pd.DataFrame(new_rows).set_index("_df_idx")
for col in pmi_df.columns:
    corrected.loc[has_logits, col] = pmi_df[col]

print("PMI-corrected signal_direction distribution:")
print(corrected["signal_direction"].value_counts(dropna=False).to_string())
print("PMI-corrected direction distribution:")
print(corrected["direction"].value_counts(dropna=False).to_string())


In [ ]:
# Cell 5 - Fetch realized returns with retry + throttling
import time
import yfinance as yf

_price_cache = {}

def _fetch_return(ticker: str, start_date: str, end_date: str):
    key = (ticker, start_date, end_date)
    if key in _price_cache:
        return _price_cache[key]
    last_error = ""
    for attempt in range(PRICE_FETCH_RETRIES):
        try:
            hist = yf.download(
                tickers=ticker,
                start=start_date,
                end=end_date,
                interval="1d",
                auto_adjust=True,
                progress=False,
                threads=False,
            )
            if hist is None or hist.empty:
                last_error = "empty_history"
            else:
                if isinstance(hist.columns, pd.MultiIndex):
                    hist.columns = hist.columns.get_level_values(0)
                if "Close" not in hist.columns:
                    last_error = "missing_close"
                else:
                    closes = hist["Close"].dropna()
                    if closes.empty:
                        last_error = "empty_closes"
                    else:
                        first, last = float(closes.iloc[0]), float(closes.iloc[-1])
                        if first == 0.0:
                            last_error = "zero_first_close"
                        else:
                            ret = (last - first) / first
                            _price_cache[key] = (ret, "")
                            return _price_cache[key]
        except Exception as exc:
            last_error = f"exception: {exc}"
        time.sleep(PRICE_FETCH_SLEEP_SEC * (attempt + 1))
    _price_cache[key] = (None, last_error or "unknown")
    return _price_cache[key]

active = corrected[corrected["signal_direction"].notna()].copy()
if not REFRESH_EXISTING_PRICES and "realized_return" in active.columns:
    already_have_price = active["realized_return"].notna()
else:
    already_have_price = pd.Series([False] * len(active), index=active.index)

realized_returns = []
price_error_reasons = []
fetch_ok = 0
for i, (_, row) in enumerate(active.iterrows()):
    if already_have_price.iloc[i]:
        realized_returns.append(float(row["realized_return"]))
        price_error_reasons.append("")
        fetch_ok += 1
    else:
        ret, reason = _fetch_return(str(row["ticker"]), str(row["start_date"]), str(row["end_date"]))
        realized_returns.append(ret)
        price_error_reasons.append(reason)
        if ret is not None:
            fetch_ok += 1
    if (i + 1) % 50 == 0:
        print(f"  price fetch {i+1}/{len(active)}  ok={fetch_ok}")
    if (i + 1) % PRICE_FETCH_BATCH_SIZE == 0:
        print(f"  pausing {PRICE_FETCH_BATCH_PAUSE_SEC}s to avoid rate-limit bursts...")
        time.sleep(PRICE_FETCH_BATCH_PAUSE_SEC)

print(f"Price fetch done: {fetch_ok}/{len(active)} valid returns.")


In [ ]:
# Cell 6 - Assemble final DataFrame and compute metrics
import math

_POS_MAP = {"long": 1, "short": -1, "neutral": 0}

def _direction_from_return(r, threshold=0.001):
    if r > threshold:
        return "up"
    if r < -threshold:
        return "down"
    return "neutral"

active = active.copy()
active["realized_return"] = realized_returns
active["price_fetch_error_reason"] = price_error_reasons
active["position"] = active["signal_direction"].map(_POS_MAP).fillna(0).astype(int)
active["strategy_return"] = [
    _POS_MAP.get(row["signal_direction"], 0) * row["realized_return"]
    if row["realized_return"] is not None and pd.notna(row["realized_return"])
    else np.nan
    for _, row in active.iterrows()
]
active["skipped_reason"] = [
    "" if r is not None and pd.notna(r) else "price_fetch_failed"
    for r in realized_returns
]

no_signal = corrected[corrected["signal_direction"].isna()].copy()
no_signal["skipped_reason"] = no_signal.get("skipped_reason", pd.Series(index=no_signal.index)).fillna("signal_failed")
if "price_fetch_error_reason" not in no_signal.columns:
    no_signal["price_fetch_error_reason"] = ""

final = pd.concat([active, no_signal], ignore_index=True)
successful = final[final["skipped_reason"] == ""].copy()
total = len(final)
ok = len(successful)
skipped = total - ok

if ok > 0:
    successful["realized_direction"] = successful["realized_return"].astype(float).apply(_direction_from_return)
    successful["signal_mkt_dir"] = successful["signal_direction"].map({"long": "up", "short": "down", "neutral": "neutral"})
    dir_acc = float((successful["signal_mkt_dir"] == successful["realized_direction"]).mean())
    longs = successful[successful["signal_direction"] == "long"]
    shorts = successful[successful["signal_direction"] == "short"]
    long_acc = float((longs["realized_direction"] == "up").mean()) if len(longs) else float("nan")
    short_acc = float((shorts["realized_direction"] == "down").mean()) if len(shorts) else float("nan")
    sr = successful["strategy_return"].astype(float)
    mu = float(sr.mean())
    std = float(sr.std(ddof=0))
    sharpe = (mu / std) * math.sqrt(52) if std > 0 else float("nan")
    total_pnl = float(sr.sum())
    fingpt_dir = successful["fingpt_label"].map({"up": "long", "down": "short", "neutral": "neutral"})
    vs_fingpt = float((successful["signal_direction"] == fingpt_dir).mean())
else:
    dir_acc = long_acc = short_acc = sharpe = total_pnl = vs_fingpt = float("nan")
    mu = std = 0.0

metrics = {
    "total_rows": total,
    "successful_rows": ok,
    "skip_rate": (skipped / total) if total else 0.0,
    "direction_accuracy": dir_acc,
    "long_accuracy": long_acc,
    "short_accuracy": short_acc,
    "mean_strategy_return": mu,
    "std_strategy_return": std,
    "annualized_sharpe": sharpe,
    "total_pnl": total_pnl,
    "vs_fingpt_accuracy": vs_fingpt,
}

print("=" * 50)
print("BACKTEST METRICS (PMI-corrected signals)")
print("=" * 50)
for k, v in metrics.items():
    print(f"  {k:<28}: {v:.4f}" if isinstance(v, float) else f"  {k:<28}: {v}")
print()
print("Signal direction breakdown (successful rows):")
if ok > 0:
    print(successful["signal_direction"].value_counts(dropna=False).to_string())
print()
print("Top price fetch failures:")
print(final["price_fetch_error_reason"].fillna("").replace("", "<ok>").value_counts().head(10).to_string())


In [ ]:
# Cell 7 - Save corrected CSV
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
alpha_slug = str(PMI_ALPHA).replace(".", "_").replace("-", "neg_")
out_path = os.path.join(OUT_DIR, f"backtest_pmi_alpha_{alpha_slug}_{timestamp}.csv")
final.drop(columns=["logits_list"], errors="ignore").to_csv(out_path, index=False)
print(f"Saved {len(final)} rows to {out_path}")
